# Flight Delays: preparazione e integrità
Prepariamo un unico contratto dati per i tre database, senza connessioni o ingestione. Il CSV originale contiene **31 colonne e nessuna singola chiave univoca**. Aggiungiamo `flight_id` da 1 a 5.819.079, seguendo i record nell'ordine originale.
La sorgente Kaggle `usdot/flight-delays`, versione 1, ha 592.406.591 byte (circa 564,96 MiB). Le righe identiche non vengono eliminate. I subset di 100k e 1M sono prefissi del completo, non nuovi campionamenti né sorgenti complete.
`Run All` mostra la configurazione senza preparare dati. Abilitare `ABILITA_PREPARAZIONE=True` solo per il setup del dataset, non durante la DEMO.


In [ ]:
from pathlib import Path
import json
import logging
import sys
import yaml

# Jupyter può partire dalla root o dalla cartella notebooks.
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'config' / 'benchmark.yaml').is_file():
    raise RuntimeError('Aprire il notebook nel container del progetto o dalla root del repository.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.preprocessing import acquire_source, prepare_dataset
from src.validation import iter_prepared_chunks, validate_bundle
from src.schema import PREPARED_COLUMNS, SOURCE_SHA256

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
config = yaml.safe_load((ROOT / 'config' / 'benchmark.yaml').read_text(encoding='utf-8'))
settings = config['preprocessing']
RAW = ROOT / config['paths']['raw_csv']
OUTPUT = ROOT / config['paths']['prepared'] / settings['output_name']
SIZES = tuple(config['dataset_sizes'])
CHUNK_SIZE = settings['chunk_size']
OFFLINE = settings['offline']
FORCE = settings['force']
print({'source': str(RAW), 'output': str(OUTPUT), 'sizes': SIZES, 'chunk_size': CHUNK_SIZE, 'offline': OFFLINE, 'force': FORCE})
print('SHA-256 atteso della sorgente:', SOURCE_SHA256)
ABILITA_PREPARAZIONE = False  # La preparazione completa richiede tempo e spazio.


## 1. Acquisizione esplicita e sorgente immutabile
Per default `offline: true`: se manca `data/raw/flights.csv`, questa cella si ferma con istruzioni, senza rete. Copiare prima il CSV completo, oppure impostare consapevolmente `offline: false` in YAML per scaricare il solo file dalla versione 1 tramite KaggleHub. La copia persistente elimina la dipendenza dalla cache.
Un file già presente non viene mai sovrascritto. SHA-256 e dimensioni sono registrati in `flights.source.json`; un checksum diverso dalla versione attesa causa un errore, non una correzione automatica.

In [ ]:
if ABILITA_PREPARAZIONE is True:
    provenance = acquire_source(RAW, offline=OFFLINE)
    print(json.dumps(provenance, indent=2, ensure_ascii=False))
else:
    print('Operazione disabilitata: ABILITA_PREPARAZIONE=False.')


## 2. Streaming, tipi e pubblicazione del bundle
`prepare_dataset` legge al massimo `CHUNK_SIZE` record originali per blocco, verifica i tipi e assegna il contatore fuori dal ciclo dei chunk. Conserva esattamente le 31 celle originali e aggiunge una sola colonna `flight_id`.
CSV UTF-8, intestazione `flight_id,YEAR,...,WEATHER_DELAY`, terminatori LF. Le celle vuote rappresentano `None` nel lettore tipizzato (`NULL` SQL / `null` Mongo). Zero e stringhe come `NA` restano valori distinti. Gli orari `HHMM` rimangono stringhe, inclusi gli zeri iniziali e `2400`; durate e ritardi sono interi nullable. I flag restano interi 0/1.
Ogni CSV viene riletto e confrontato cella per cella con il prefisso originale. La sequenza esatta 1..N prova unicità, assenza di ID nulli e ordine senza mantenere tutte le chiavi in RAM. Solo alla fine il bundle diventa visibile come directory validata. Anche preparando un solo subset si scansiona tutta la sorgente.

In [ ]:
if ABILITA_PREPARAZIONE is True:
    result = prepare_dataset(RAW, OUTPUT, sizes=SIZES, chunk_size=CHUNK_SIZE, force=FORCE)
    manifest = result['manifest']
    print('Checkpoint riutilizzato:', result['skipped'])
    for item in manifest['artifacts']:
        print(item['file'], '| righe:', item['rows'], '| ID:', item['flight_id_min'], '..', item['flight_id_max'], '| SHA-256:', item['sha256'])
    print('Statistiche sorgente:', json.dumps(manifest['source']['statistics'], indent=2))
else:
    print('Operazione disabilitata: ABILITA_PREPARAZIONE=False.')


## 3. Risorse osservate, non misure dei database
Il manifest registra il tempo di checksum, scrittura e validazione. L'acquisizione e il rename finale sono esclusi. La memoria è il massimo RSS del processo campionato ogni 100 ms con psutil, non la memoria di tutta Docker né un massimo matematico garantito. Nel notebook comprende anche il processo kernel.
Se compare `skipped=True`, le misure nel manifest sono quelle della preparazione precedente; `invocation_metrics` descrive invece il controllo checksum appena eseguito. Non mescolare questi due tempi.

In [ ]:
if ABILITA_PREPARAZIONE is True:
    print('Preparazione salvata:', json.dumps(manifest['metrics'], indent=2))
    print('Chiamata corrente:', json.dumps(result['invocation_metrics'], indent=2))
    print('Picco RSS della preparazione (MiB):', round(manifest['metrics']['sampled_peak_rss_bytes'] / 2**20, 2))
else:
    print('Operazione disabilitata: ABILITA_PREPARAZIONE=False.')


## 4. Verifica e lettura riutilizzabile
Lo skip verifica nuovamente SHA-256 della sorgente e di ogni output. Per ripetere tutti i controlli su celle/ID/tipi impostare `DEEP_RECHECK=True` sotto (richiede un'altra scansione dei file). L'esempio legge soltanto due record dal subset.
La mappatura SQL/BSON è salvata in `schema.json` e nel manifest: gli adapter la applicano esplicitamente, soprattutto `flight_id` -> BSON Int64 come `_id`. Qui non vengono creati schemi o documenti nei DB.

In [ ]:
if ABILITA_PREPARAZIONE is True:
    DEEP_RECHECK = False
    checked = validate_bundle(OUTPUT, RAW, deep=DEEP_RECHECK, expected_sizes=tuple(sorted(SIZES)))
    print('Verifica completata:', checked['status'])
    sample_path = OUTPUT / manifest['artifacts'][0]['file']
    for row in next(iter_prepared_chunks(sample_path, chunk_size=2)):
        print(dict(zip(PREPARED_COLUMNS, row)))
else:
    print('Operazione disabilitata: ABILITA_PREPARAZIONE=False.')


## Riutilizzo e gestione degli errori
Una seconda esecuzione con gli stessi subset riusa soltanto un checkpoint valido, senza riscrivere i CSV. Se un checksum differisce si ferma. `force: true` è una rigenerazione esplicita: conserva il vecchio bundle in una directory `.flights-v1.backup-...` e richiede spazio anche per la nuova copia. Non cancella l'originale.
Un processo interrotto non produce un checkpoint valido. Lock o staging residui vanno esaminati soltanto dopo aver verificato che non esista una preparazione ancora attiva. Nessuna ripresa automatica da un punto ambiguo del CSV.
La DEMO usa il dataset persistente: non eseguire preprocessing o download durante la dimostrazione. Il requisito dimensionale riguarda il sorgente completo, non il subset da 100k/1M. Qui non si eseguono ingestione o query.